# Dealer Positioning Color — 2026-07-02 (Thursday)

Single-day event study: where model-labelled D2C flow landed across the STIR curve,
annotated against intraday SFR futures prices.

> **Direction is a price-implied inference, not observed party identity.**
> Coverage reflects SFR data availability at each print's timestamp.
> Labels are "model-labelled D2C flow proxy" per the external audit.

In [ ]:
import datetime
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import psycopg2
import pytz

from SDRUtils._swappulse_scripts.ingest_usdswaps_tape import resolve_pg_url
from SDRUtils.stir_flow.ladder_state import ladder_at, ladder_grid
from MDP.STIRFutures.STIRFutureMDP import (
    STIRFutureMDP, _resolve_aliases_bulk, _to_barchart_symbol,
    _normalize_symbol, _from_barchart_symbol,
)

warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")

NY = pytz.timezone("America/New_York")
CHI = pytz.timezone("America/Chicago")
TARGET_DATE = datetime.date(2026, 7, 2)

# dataviz palette: categorical slots 1,3,4,5,6,7 (skip 2=green, 8=red to avoid
# collision with direction arrows)
CONTRACT_COLORS = ["#2a78d6", "#e87ba4", "#eda100", "#1baf7a", "#eb6834", "#4a3aa7"]
RECEIVED_COLOR = "#008300"
PAID_COLOR = "#e34948"
SURFACE_LIGHT = "#fcfcfb"
GRID_COLOR = "#e1e0d9"
TEXT_SECONDARY = "#52514e"
TEXT_MUTED = "#898781"

In [ ]:
url = resolve_pg_url()
conn = psycopg2.connect(url)

directions = pd.read_sql("""
    SELECT unit_key, execution_timestamp, dealer_direction,
           classification_method, direction_confidence, p_flip,
           structure_dv01, dv01, fixed_rate, curve_mid,
           rate_index_clean, trade_type, notional, is_off_market
    FROM arbs_stir_direction_v1
    WHERE execution_timestamp::date = '2026-07-02'
      AND dealer_direction IN ('PAID', 'RECEIVED')
    ORDER BY execution_timestamp
""", conn)

prints_df = pd.read_sql("""
    SELECT unit_key, bucket_space, bucket_key, delta_dv01,
           execution_timestamp, visibility_timestamp,
           p_flip, direction_confidence, curve_suspect_trade,
           is_block, dv01
    FROM arbs_stir_ladder_prints_v1
    WHERE execution_timestamp::date = '2026-07-02'
    ORDER BY visibility_timestamp
""", conn)

conn.close()

assert len(directions) > 0, "No classifications found — run backfill_stir_direction first"
assert len(prints_df) > 0, "No ladder prints — run backfill_stir_ladder --phase project first"

print(f"Classified prints: {len(directions)}")
print(directions["dealer_direction"].value_counts().to_string())
print(f"\nLadder prints: {len(prints_df)}")
print(prints_df["bucket_space"].value_counts().to_string())

In [ ]:
aliases = _resolve_aliases_bulk([f"SFRCM{i}" for i in range(1, 7)], TARGET_DATE)
contracts = [t for tlist in aliases.values() for t in tlist]
alias_to_contract = {a: tl[0] for a, tl in aliases.items()}
print(f"Front 6 contracts: {contracts}")

ts_noon = NY.localize(datetime.datetime(2026, 7, 2, 12, 0, 0))

mdp = STIRFutureMDP(source="BARCHART_STIRF-RL")
with mdp:
    price_df = mdp._fetch_barchart_timeseries(
        contracts, ts_noon, show_tqdm=True, interval=1, full_day_intraday=True
    )

    # Volume: separate call via internal fetcher
    ts_chi = ts_noon.astimezone(CHI)
    session_open = ts_chi.replace(hour=17, minute=0, second=0, microsecond=0)
    if ts_chi < session_open:
        session_open -= datetime.timedelta(days=1)
    session_close = session_open + datetime.timedelta(hours=23)

    bcf = mdp._get_barchart_fetcher(required_concurrency=8)
    barchart_syms = [_to_barchart_symbol(_normalize_symbol(t) or t) for t in contracts]
    vol_df = bcf.barchart_timeseries_api(
        barchart_symbols=barchart_syms,
        start_date=session_open,
        end_date=session_close,
        interval=1,
        one_df=True,
        merge_val_col="Volume",
        show_tqdm=True,
    )
    vol_df.columns = [_from_barchart_symbol(c) for c in vol_df.columns]
    bcf.close()

# Focus on US trading hours (8am-5pm ET)
us_start = NY.localize(datetime.datetime(2026, 7, 2, 8, 0))
us_end = NY.localize(datetime.datetime(2026, 7, 2, 17, 0))

if price_df.index.tz is not None:
    us_start = us_start.astimezone(price_df.index.tz)
    us_end = us_end.astimezone(price_df.index.tz)

price_us = price_df.loc[us_start:us_end]
vol_us = vol_df.loc[us_start:us_end] if not vol_df.empty else pd.DataFrame()

print(f"\nPrice bars: {price_us.shape}")
print(f"Volume bars: {vol_us.shape}")
print(f"Time range: {price_us.index.min()} → {price_us.index.max()}")

In [ ]:
# Per-contract price with classified flow annotations
# FUTURES-space bucket_keys use SFR prefix (e.g. SFRU26), prices use SR3 (e.g. SR3U26)
fut_prints = prints_df[prints_df["bucket_space"] == "FUTURES"].copy()
fut_prints["visibility_timestamp"] = pd.to_datetime(fut_prints["visibility_timestamp"])

# Map SFR bucket keys to SR3 contract symbols
fut_prints["contract"] = fut_prints["bucket_key"].str.replace("SFR", "SR3", regex=False)

n_contracts = len(contracts)
fig = make_subplots(
    rows=n_contracts, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[f"{c} ({list(aliases.keys())[i]})" for i, c in enumerate(contracts)],
)

for i, contract in enumerate(contracts):
    row = i + 1
    color = CONTRACT_COLORS[i]

    if contract in price_us.columns:
        series = price_us[contract].dropna()
        fig.add_trace(go.Scatter(
            x=series.index, y=series.values,
            mode="lines", line=dict(width=2, color=color),
            name=contract, showlegend=(i == 0),
            hovertemplate=f"{contract}: %{{y:.3f}}<extra></extra>",
        ), row=row, col=1)

    cp = fut_prints[fut_prints["contract"] == contract]
    if cp.empty:
        continue

    if contract not in price_us.columns:
        continue
    price_series = price_us[contract].dropna()

    for direction, marker_symbol, arrow_color, label in [
        ("recv", "triangle-up", RECEIVED_COLOR, "RECEIVED"),
        ("paid", "triangle-down", PAID_COLOR, "PAID"),
    ]:
        mask = cp["delta_dv01"] > 0 if direction == "recv" else cp["delta_dv01"] < 0
        subset = cp[mask]
        if subset.empty:
            continue

        vis_ts = subset["visibility_timestamp"]
        if price_series.index.tz is not None and vis_ts.dt.tz is None:
            vis_ts = vis_ts.dt.tz_localize("UTC").dt.tz_convert(price_series.index.tz)
        elif price_series.index.tz is not None:
            vis_ts = vis_ts.dt.tz_convert(price_series.index.tz)

        y_vals = []
        for t in vis_ts:
            idx = price_series.index.get_indexer([t], method="nearest")
            y_vals.append(float(price_series.iloc[idx[0]]) if idx[0] != -1 else np.nan)

        sizes = np.clip(subset["delta_dv01"].abs().values * 15, 6, 30)

        fig.add_trace(go.Scatter(
            x=vis_ts, y=y_vals,
            mode="markers",
            marker=dict(
                symbol=marker_symbol, size=sizes, color=arrow_color,
                line=dict(width=1, color="white"),
            ),
            name=label, showlegend=(i == 0),
            hovertemplate=(
                f"{label}<br>"
                f"|DV01|: %{{customdata[0]:.2f}}<br>"
                f"conf: %{{customdata[1]}}<br>"
                f"vis: %{{x}}<extra>{contract}</extra>"
            ),
            customdata=list(zip(
                subset["delta_dv01"].abs().values,
                subset["direction_confidence"].fillna("?").values,
            )),
        ), row=row, col=1)

fig.update_layout(
    height=250 * n_contracts,
    title_text="SFR Futures Price + Model-Labelled D2C Flow (FUTURES space)",
    plot_bgcolor=SURFACE_LIGHT,
    paper_bgcolor="white",
    font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    hovermode="x unified",
)
for i in range(n_contracts):
    fig.update_yaxes(gridcolor=GRID_COLOR, gridwidth=1, row=i+1, col=1)
    fig.update_xaxes(gridcolor=GRID_COLOR, gridwidth=1, row=i+1, col=1)

fig.show()

In [ ]:
# Cumulative ladder evaluated every 5 minutes through the trading day
eval_start = NY.localize(datetime.datetime(2026, 7, 2, 8, 0))
eval_end = NY.localize(datetime.datetime(2026, 7, 2, 17, 0))
eval_ts = pd.date_range(eval_start, eval_end, freq="5min")

# Diverging colorscale: blue (RECEIVED/long) ↔ gray ↔ red (PAID/short)
# Uses palette diverging pair for CVD safety (heatmap has no secondary encoding)
DIVERGING_SCALE = [
    [0, PAID_COLOR],
    [0.5, "#f0efec"],
    [1, "#2a78d6"],
]

for space, title in [("MEETING", "MEETING-space (FOMC dates)"), ("FUTURES", "FUTURES-space (contracts)")]:
    grid = ladder_grid(prints_df, eval_ts, space=space)
    if grid.empty or grid.abs().sum().sum() == 0:
        print(f"No {space}-space data")
        continue

    grid = grid[sorted(grid.columns)]
    z_max = grid.abs().max().max()

    fig = go.Figure(data=go.Heatmap(
        z=grid.values.T,
        x=grid.index,
        y=grid.columns,
        colorscale=DIVERGING_SCALE,
        zmid=0,
        zmin=-z_max if z_max else -1,
        zmax=z_max if z_max else 1,
        colorbar=dict(title="Net DV01<br>(+recv / −paid)"),
        hovertemplate="%{y}<br>%{x}<br>Net DV01: %{z:.2f}<extra></extra>",
    ))

    fig.update_layout(
        title=f"Cumulative Positioning — {title}",
        xaxis_title="Time (ET)",
        yaxis_title="Bucket",
        height=max(400, len(grid.columns) * 40 + 150),
        plot_bgcolor=SURFACE_LIGHT,
        paper_bgcolor="white",
        font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
        yaxis=dict(type="category"),
    )
    fig.show()

In [ ]:
# EOD net positioning per bucket
eod_ts = eval_ts[-1]

summary_rows = []
for space in ["MEETING", "FUTURES"]:
    eod = ladder_at(prints_df, eod_ts, space=space)
    sp = prints_df[prints_df["bucket_space"] == space]
    for bucket, net in eod.items():
        bp = sp[sp["bucket_key"] == bucket]
        recv = bp.loc[bp["delta_dv01"] > 0, "delta_dv01"].sum()
        paid = bp.loc[bp["delta_dv01"] < 0, "delta_dv01"].sum()
        summary_rows.append(dict(
            space=space, bucket=bucket, n_prints=len(bp),
            received_dv01=round(recv, 2), paid_dv01=round(paid, 2),
            net_dv01=round(net, 2),
        ))

summary = pd.DataFrame(summary_rows)
print("=== EOD Net Positioning (decay-weighted, expected) ===")
display(summary)

# Trade log: join direction details with print timestamps
log = directions.merge(
    prints_df[["unit_key", "visibility_timestamp", "bucket_space", "bucket_key", "delta_dv01"]],
    on="unit_key", how="inner",
).sort_values("execution_timestamp")

log_display = log[[
    "execution_timestamp", "visibility_timestamp", "dealer_direction",
    "classification_method", "direction_confidence", "bucket_space",
    "bucket_key", "delta_dv01", "structure_dv01", "trade_type", "rate_index_clean",
]].copy()
log_display.columns = [
    "exec_ts", "vis_ts", "direction", "method", "confidence",
    "space", "bucket", "delta_dv01", "struct_dv01", "type", "index",
]

print(f"\n=== Trade Log ({len(log_display)} print-bucket rows from {log_display['exec_ts'].nunique()} classified trades) ===")
with pd.option_context("display.max_rows", 200, "display.max_columns", 20, "display.width", 200):
    display(log_display)

In [ ]:
# Per-contract volume profile
if vol_us.empty:
    print("No volume data available")
else:
    available = [c for c in contracts if c in vol_us.columns]
    n = len(available)
    if n == 0:
        print("No volume data for target contracts")
    else:
        fig = make_subplots(
            rows=n, cols=1, shared_xaxes=True,
            vertical_spacing=0.03,
            subplot_titles=available,
        )
        for i, contract in enumerate(available):
            vs = vol_us[contract].dropna()
            fig.add_trace(go.Bar(
                x=vs.index, y=vs.values,
                marker_color=CONTRACT_COLORS[contracts.index(contract)],
                name=contract, showlegend=False,
                hovertemplate=f"{contract} vol: %{{y:,.0f}}<extra></extra>",
            ), row=i+1, col=1)

        fig.update_layout(
            height=200 * n,
            title_text="1-Min Volume by Contract",
            plot_bgcolor=SURFACE_LIGHT,
            paper_bgcolor="white",
            font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
        )
        for i in range(n):
            fig.update_yaxes(gridcolor=GRID_COLOR, gridwidth=1, row=i+1, col=1)
            fig.update_xaxes(gridcolor=GRID_COLOR, gridwidth=1, row=i+1, col=1)
        fig.show()